In [1]:
import pandas as pd

df = pd.read_csv("data/deepfake.csv")
print(df.shape)
df.head()

(6557, 17)


,image_id,image_url,label,label_numeric,category,gender,age_group,source,fake_method,image_quality,resolution,confidence_score,detection_difficulty,dataset_split,date_collected,version,year
0,1,https://images.unsplash.com/photo-172422561835...,REAL,1,Authentic,Female,18-25,Unsplash,NaN,High,1080x1080,0.96,Easy,train,2026-04-24,v2.0,2026
1,2,https://images.unsplash.com/photo-172422561873...,REAL,1,Authentic,Male,50+,Unsplash,NaN,High,1080x1080,0.97,Easy,train,2026-04-04,v2.0,2026
2,3,https://images.unsplash.com/photo-172422561834...,REAL,1,Authentic,Female,36-50,Unsplash,NaN,High,1080x1080,0.95,Easy,test,2026-02-07,v2.0,2026
3,4,https://images.unsplash.com/photo-172422561855...,REAL,1,Authentic,Female,26-35,Unsplash,NaN,High,1080x1080,0.89,Easy,train,2026-02-20,v2.0,2026
4,5,https://images.unsplash.com/photo-172422561823...,REAL,1,Authentic,Male,36-50,Unsplash,NaN,High,1080x1080,0.86,Easy,train,2026-04-12,v2.0,2026


In [2]:
df.dtypes

image_id                  int64
image_url                   str
label                       str
label_numeric             int64
category                    str
gender                      str
age_group                   str
source                      str
fake_method                 str
image_quality               str
resolution                  str
confidence_score        float64
detection_difficulty        str
dataset_split               str
date_collected              str
version                     str
year                      int64
dtype: object

In [3]:
df["label"].value_counts()

label
FAKE    3767
REAL    2790
Name: count, dtype: int64

In [4]:
df["domain"] = df["image_url"].str.extract(r"https?://([^/]+)/")
pd.crosstab(df["domain"], df["label"])

label,FAKE,REAL
domain,,
api.dicebear.com,2499,0
api.multiavatar.com,1000,0
i.pravatar.cc,70,0
images.unsplash.com,0,2790
randomuser.me,198,0


In [5]:
pd.crosstab(df["label"], df["detection_difficulty"])

detection_difficulty,Easy,Hard,Medium
label,,,
FAKE,0,2461,1306
REAL,2790,0,0


In [6]:
pd.crosstab(df["label"], df["resolution"])

resolution,1024x1024,1080x1080
label,,
FAKE,3767,0
REAL,0,2790


In [7]:
real = df[df["label"] == "REAL"].copy()
real["photo_id"] = real["image_url"].str.extract(r"(photo-[0-9a-f-]+)")[0]
repeated = real[real["photo_id"].duplicated(keep=False)]
conflicts = repeated.groupby("photo_id")[["gender", "age_group"]].nunique()
print(len(real), "rows,", real["photo_id"].nunique(), "unique photos")
print(len(conflicts), "photos repeated:", (conflicts["gender"] > 1).sum(), "with conflicting gender,",
      (conflicts["age_group"] > 1).sum(), "with conflicting age group")

2790 rows, 2349 unique photos
397 photos repeated: 262 with conflicting gender, 295 with conflicting age group


## Audit findings
6557 rows, 17 columns. 2790 real images, 3767 fake images. The CSV contains image URLs, not images

### None of the images are AI generated deepfakes
Most of them are cartoon avatars, some are images of real people, and many of the requests return an HTTP 403 Forbidden response

### Some of the metadata gives away the label
The detection difficulty column lists all of the real images as 'Easy' and every fake image as 'Medium' or 'Hard' with no overlap.
The resolution column lists all real images as 1080x1080 and every fake image as 1024x1024

### The real images contain duplicates
Of the roughly 2800 real images, about 15% were duplicates.

### Conclusion
This dataset is unfit for the purpose of creating a model that can adequately detect AI generated deepfake images. The real images can still be used as long as the duplicates are excluded. It is necessary to create another set of fake images. (See '02_build_dataset')